## Carga de los datos

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

# Identificar raíz del proyecto
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        ruta
        for ruta in (cwd, *cwd.parents)
        if (ruta / "data" / "interim").is_dir()
        and (ruta / "notebooks").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto."
    )

# Directorios
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

# Cargar información
panel = pd.read_parquet(
    INTERIM_DIR / "panel_bancario_2026_09.parquet"
)

# Normalizar fechas
panel["fecha"] = pd.to_datetime(panel["fecha"])

# Ordenar
panel = (
    panel
    .sort_values(["banco", "fecha"])
    .reset_index(drop=True)
)

# Comprobar integridad
assert not panel.duplicated(["banco", "fecha"]).any()

print("Dimensiones:", panel.shape)
print("Instituciones:", panel["banco"].nunique())

display(panel.head())

Dimensiones: (5972, 11)
Instituciones: 23


,fecha,banco,solvencia,liquidez,roa,roe,cartera_depositos,cartera_bruta,cartera_improductiva,morosidad_total,cobertura
0,2011-07-01,BANCO ATLANTIDA S.A.,NaN,0.0,0.019541,0.091531,0.0,31872.72431,828.74366,0.026002,0.776931
1,2011-08-01,BANCO ATLANTIDA S.A.,0.25177,0.0,0.018258,0.085764,0.0,33162.49816,728.98974,0.021982,1.002009
2,2011-09-01,BANCO ATLANTIDA S.A.,0.28814,0.0,0.017008,0.081672,0.0,34827.65249,822.65943,0.023621,1.003828
3,2011-10-01,BANCO ATLANTIDA S.A.,0.26950,0.0,0.016025,0.073058,0.0,36890.47681,909.87157,0.024664,1.026205
4,2011-11-01,BANCO ATLANTIDA S.A.,0.26057,0.0,0.015737,0.070117,0.0,38955.55403,1020.61617,0.026200,1.007305


## Construccion del target de los 3 meses siguientes

In [3]:
# ==========================================
# CONSTRUCCIÓN DE TARGETS MULTIHORIZONTE
# ==========================================

HORIZONTES = [1, 2, 3]

# Partir del panel histórico completo
df_features = (
    panel
    .sort_values(["banco", "fecha"])
    .reset_index(drop=True)
    .copy()
)

# Convertir morosidad a puntos porcentuales
df_features["morosidad_pp"] = (
    df_features["morosidad_total"] * 100
)

# Verificar integridad
assert not df_features.duplicated(
    ["banco", "fecha"]
).any()

assert df_features["fecha"].dt.is_month_start.all()


# Construir un target por horizonte
for h in HORIZONTES:

    nombre_target = f"target_morosidad_t{h}_pp"

    # Tabla auxiliar con morosidad futura
    futuro = df_features[
        ["banco", "fecha", "morosidad_pp"]
    ].copy()

    # Desplazar las fechas h meses hacia atrás
    futuro["fecha"] = (
        futuro["fecha"]
        - pd.DateOffset(months=h)
    )

    # Renombrar variable objetivo
    futuro = futuro.rename(
        columns={"morosidad_pp": nombre_target}
    )

    # Incorporar target mediante banco y fecha
    df_features = df_features.merge(
        futuro,
        on=["banco", "fecha"],
        how="left",
        validate="one_to_one"
    )

    # Fecha calendario correspondiente al objetivo
    df_features[f"fecha_objetivo_t{h}"] = (
        df_features["fecha"]
        + pd.DateOffset(months=h)
    )


# Visualizar resultados
columnas = [
    "fecha",
    "banco",
    "morosidad_pp",
    "target_morosidad_t1_pp",
    "target_morosidad_t2_pp",
    "target_morosidad_t3_pp"
]

display(df_features[columnas].head(10))

,fecha,banco,morosidad_pp,target_morosidad_t1_pp,target_morosidad_t2_pp,target_morosidad_t3_pp
0,2011-07-01,BANCO ATLANTIDA S.A.,2.600166,2.198235,2.362087,2.466413
1,2011-08-01,BANCO ATLANTIDA S.A.,2.198235,2.362087,2.466413,2.619950
2,2011-09-01,BANCO ATLANTIDA S.A.,2.362087,2.466413,2.619950,2.230437
3,2011-10-01,BANCO ATLANTIDA S.A.,2.466413,2.619950,2.230437,2.682136
4,2011-11-01,BANCO ATLANTIDA S.A.,2.619950,2.230437,2.682136,3.232366
5,2011-12-01,BANCO ATLANTIDA S.A.,2.230437,2.682136,3.232366,2.989239
6,2012-01-01,BANCO ATLANTIDA S.A.,2.682136,3.232366,2.989239,3.347692
7,2012-02-01,BANCO ATLANTIDA S.A.,3.232366,2.989239,3.347692,3.623611
8,2012-03-01,BANCO ATLANTIDA S.A.,2.989239,3.347692,3.623611,4.045769
9,2012-04-01,BANCO ATLANTIDA S.A.,3.347692,3.623611,4.045769,3.914882


## Comprobar las fronteras temporales

In [4]:
# ==========================================
# CARACTERÍSTICAS TEMPORALES
# ==========================================

features_temporales = df_features.copy()

# Rezagos históricos
REZAGOS = [1, 2, 3, 6, 12]

for k in REZAGOS:

    # Tabla auxiliar
    historico = df_features[
        ["banco", "fecha", "morosidad_pp"]
    ].copy()

    # Desplazar fechas hacia adelante
    historico["fecha"] = (
        historico["fecha"]
        + pd.DateOffset(months=k)
    )

    # Nombre de la característica
    nombre_lag = f"morosidad_lag_{k}_pp"

    historico = historico.rename(
        columns={"morosidad_pp": nombre_lag}
    )

    # Incorporar rezago
    features_temporales = features_temporales.merge(
        historico,
        on=["banco", "fecha"],
        how="left",
        validate="one_to_one"
    )


# ==========================================
# VARIACIONES HISTÓRICAS
# ==========================================

# Variación del último mes
features_temporales["delta_morosidad_1m_pp"] = (
    features_temporales["morosidad_pp"]
    - features_temporales["morosidad_lag_1_pp"]
)

# Variación acumulada de tres meses
features_temporales["delta_morosidad_3m_pp"] = (
    features_temporales["morosidad_pp"]
    - features_temporales["morosidad_lag_3_pp"]
)


# Visualizar
display(
    features_temporales[
        [
            "fecha",
            "banco",
            "morosidad_pp",
            "morosidad_lag_1_pp",
            "morosidad_lag_3_pp",
            "delta_morosidad_1m_pp",
            "delta_morosidad_3m_pp",
            "target_morosidad_t1_pp",
            "target_morosidad_t2_pp",
            "target_morosidad_t3_pp"
        ]
    ].head(15)
)

,fecha,banco,morosidad_pp,morosidad_lag_1_pp,morosidad_lag_3_pp,delta_morosidad_1m_pp,delta_morosidad_3m_pp,target_morosidad_t1_pp,target_morosidad_t2_pp,target_morosidad_t3_pp
0,2011-07-01,BANCO ATLANTIDA S.A.,2.600166,NaN,NaN,NaN,NaN,2.198235,2.362087,2.466413
1,2011-08-01,BANCO ATLANTIDA S.A.,2.198235,2.600166,NaN,-0.401930,NaN,2.362087,2.466413,2.619950
2,2011-09-01,BANCO ATLANTIDA S.A.,2.362087,2.198235,NaN,0.163852,NaN,2.466413,2.619950,2.230437
3,2011-10-01,BANCO ATLANTIDA S.A.,2.466413,2.362087,2.600166,0.104326,-0.133753,2.619950,2.230437,2.682136
4,2011-11-01,BANCO ATLANTIDA S.A.,2.619950,2.466413,2.198235,0.153537,0.421715,2.230437,2.682136,3.232366
5,2011-12-01,BANCO ATLANTIDA S.A.,2.230437,2.619950,2.362087,-0.389514,-0.131650,2.682136,3.232366,2.989239
6,2012-01-01,BANCO ATLANTIDA S.A.,2.682136,2.230437,2.466413,0.451699,0.215723,3.232366,2.989239,3.347692
7,2012-02-01,BANCO ATLANTIDA S.A.,3.232366,2.682136,2.619950,0.550230,0.612416,2.989239,3.347692,3.623611
8,2012-03-01,BANCO ATLANTIDA S.A.,2.989239,3.232366,2.230437,-0.243127,0.758803,3.347692,3.623611,4.045769
9,2012-04-01,BANCO ATLANTIDA S.A.,3.347692,2.989239,2.682136,0.358453,0.665556,3.623611,4.045769,3.914882


## Incorporar estacionalidad

In [5]:
# ==========================================
# CARACTERÍSTICAS ESTACIONALES MULTIHORIZONTE
# ==========================================

for h in HORIZONTES:

    # Fecha objetivo
    fecha_objetivo = (
        features_temporales[f"fecha_objetivo_t{h}"]
    )

    # Mes del objetivo
    mes = fecha_objetivo.dt.month

    # Guardar mes
    features_temporales[f"mes_objetivo_t{h}"] = mes

    # Codificación cíclica
    features_temporales[f"mes_sin_t{h}"] = np.sin(
        2 * np.pi * mes / 12
    )

    features_temporales[f"mes_cos_t{h}"] = np.cos(
        2 * np.pi * mes / 12
    )


# Comprobar un ejemplo
display(
    features_temporales.loc[
        (
            features_temporales["banco"] == "BP PICHINCHA"
        )
        & (
            features_temporales["fecha"]
            == "2025-09-01"
        ),
        [
            "fecha",
            "fecha_objetivo_t1",
            "fecha_objetivo_t2",
            "fecha_objetivo_t3",
            "mes_objetivo_t1",
            "mes_objetivo_t2",
            "mes_objetivo_t3",
            "mes_sin_t1",
            "mes_cos_t1",
            "mes_sin_t2",
            "mes_cos_t2",
            "mes_sin_t3",
            "mes_cos_t3"
        ]
    ]
)

,fecha,fecha_objetivo_t1,fecha_objetivo_t2,fecha_objetivo_t3,mes_objetivo_t1,mes_objetivo_t2,mes_objetivo_t3,mes_sin_t1,mes_cos_t1,mes_sin_t2,mes_cos_t2,mes_sin_t3,mes_cos_t3
4983,2025-09-01,2025-10-01,2025-11-01,2025-12-01,10,11,12,-0.866025,0.5,-0.5,0.866025,-2.449294e-16,1.0


## Validar los targets

In [6]:
# ==========================================
# VALIDACIÓN MULTIHORIZONTE
# ==========================================

print("DIMENSIONES")
print(features_temporales.shape)

print("\nINSTITUCIONES")
print(features_temporales["banco"].nunique())

print("\nTARGETS POR HORIZONTE")

for h in HORIZONTES:

    columna = f"target_morosidad_t{h}_pp"
    fecha_columna = f"fecha_objetivo_t{h}"

    disponibles = features_temporales[columna].notna().sum()
    faltantes = features_temporales[columna].isna().sum()

    # Verificar fechas objetivo
    fechas_esperadas = (
        features_temporales["fecha"]
        + pd.DateOffset(months=h)
    )

    assert (
        features_temporales[fecha_columna]
        == fechas_esperadas
    ).all()

    print(f"\nHorizonte: {h} mes(es)")
    print("Targets disponibles:", disponibles)
    print("Targets faltantes:", faltantes)


# Validaciones estructurales
assert len(features_temporales) == len(panel)

assert not features_temporales.duplicated(
    ["banco", "fecha"]
).any()

assert (
    features_temporales["target_morosidad_t1_pp"]
    .notna()
    .sum()
    == 5944
)

print("\nVALIDACIÓN ESTRUCTURAL COMPLETADA")

DIMENSIONES
(5972, 34)

INSTITUCIONES
23

TARGETS POR HORIZONTE

Horizonte: 1 mes(es)
Targets disponibles: 5944
Targets faltantes: 28

Horizonte: 2 mes(es)
Targets disponibles: 5922
Targets faltantes: 50

Horizonte: 3 mes(es)
Targets disponibles: 5900
Targets faltantes: 72

VALIDACIÓN ESTRUCTURAL COMPLETADA


## Separacion entre desarrollo y test

In [7]:
# ==========================================
# SEPARACIÓN TEMPORAL MULTIHORIZONTE
# ==========================================

TEST_INICIO = pd.Timestamp("2025-10-01")
TEST_FIN = pd.Timestamp("2026-09-01")

# Último mes utilizado para entrenar
CORTE_ENTRENAMIENTO = pd.Timestamp("2025-09-01")

# Almacenar datasets
datasets_horizontes = {}

for h in HORIZONTES:

    target = f"target_morosidad_t{h}_pp"
    fecha_target = f"fecha_objetivo_t{h}"

    # Seleccionar observaciones con target válido
    datos = features_temporales.dropna(
        subset=[target]
    ).copy()

    # Entrenamiento/desarrollo:
    # solo objetivos anteriores al período Test
    desarrollo_h = datos.loc[
        datos[fecha_target] < TEST_INICIO
    ].copy()

    # Test:
    # objetivos dentro del período reservado
    # y pronósticos emitidos desde el corte
    test_h = datos.loc[
        datos[fecha_target].between(
            TEST_INICIO,
            TEST_FIN
        )
        & (datos["fecha"] >= CORTE_ENTRENAMIENTO)
    ].copy()

    # Verificar frontera temporal
    assert (
        desarrollo_h[fecha_target].max()
        <= CORTE_ENTRENAMIENTO
    )

    assert (
        test_h["fecha"].min()
        >= CORTE_ENTRENAMIENTO
    )

    # Guardar
    datasets_horizontes[h] = {
        "desarrollo": desarrollo_h,
        "test": test_h
    }

    print(f"\nHORIZONTE {h} MES(ES)")
    print("Desarrollo:", len(desarrollo_h))
    print("Test:", len(test_h))

    print(
        "Primer target Test:",
        test_h[fecha_target].min()
    )

    print(
        "Último target Test:",
        test_h[fecha_target].max()
    )


HORIZONTE 1 MES(ES)
Desarrollo: 5668
Test: 276
Primer target Test: 2025-10-01 00:00:00
Último target Test: 2026-09-01 00:00:00

HORIZONTE 2 MES(ES)
Desarrollo: 5646
Test: 253
Primer target Test: 2025-11-01 00:00:00
Último target Test: 2026-09-01 00:00:00

HORIZONTE 3 MES(ES)
Desarrollo: 5624
Test: 230
Primer target Test: 2025-12-01 00:00:00
Último target Test: 2026-09-01 00:00:00


## Estadisticas moviles de morosidad

### Construccion de las estadisticas moviles

In [8]:
# ==========================================
# ESTADÍSTICAS MÓVILES DE MOROSIDAD
# ==========================================

VENTANAS = [3, 6]

resultados_moviles = []

for banco, datos in features_temporales.groupby("banco"):

    # Construir serie mensual por institución
    serie = (
        datos
        .set_index("fecha")["morosidad_pp"]
        .sort_index()
        .asfreq("MS")
    )

    # Tabla auxiliar
    auxiliar = pd.DataFrame({
        "banco": banco,
        "fecha": serie.index
    })

    for ventana in VENTANAS:

        # Ventana móvil
        movil = serie.rolling(
            window=ventana,
            min_periods=ventana
        )

        # Media móvil
        auxiliar[f"morosidad_media_{ventana}m_pp"] = (
            movil.mean().to_numpy()
        )

        # Desviación estándar móvil
        auxiliar[f"morosidad_std_{ventana}m_pp"] = (
            movil.std(ddof=0).to_numpy()
        )

    resultados_moviles.append(auxiliar)


# Unificar características
moviles = pd.concat(
    resultados_moviles,
    ignore_index=True
)

# Incorporarlas al dataset
features_temporales = features_temporales.merge(
    moviles,
    on=["banco", "fecha"],
    how="left",
    validate="one_to_one"
)

# Comprobar resultados
columnas_moviles = [
    "morosidad_media_3m_pp",
    "morosidad_std_3m_pp",
    "morosidad_media_6m_pp",
    "morosidad_std_6m_pp"
]

print("Dimensiones:", features_temporales.shape)

print("\nValores faltantes:")
display(
    features_temporales[columnas_moviles]
    .isna()
    .sum()
)

display(
    features_temporales[
        ["fecha", "banco", "morosidad_pp"]
        + columnas_moviles
    ].head(15)
)

Dimensiones: (5972, 38)

Valores faltantes:


morosidad_media_3m_pp     53
morosidad_std_3m_pp       53
morosidad_media_6m_pp    122
morosidad_std_6m_pp      122
dtype: int64

,fecha,banco,morosidad_pp,morosidad_media_3m_pp,morosidad_std_3m_pp,morosidad_media_6m_pp,morosidad_std_6m_pp
0,2011-07-01,BANCO ATLANTIDA S.A.,2.600166,NaN,NaN,NaN,NaN
1,2011-08-01,BANCO ATLANTIDA S.A.,2.198235,NaN,NaN,NaN,NaN
2,2011-09-01,BANCO ATLANTIDA S.A.,2.362087,2.386829,0.165017,NaN,NaN
3,2011-10-01,BANCO ATLANTIDA S.A.,2.466413,2.342245,0.110378,NaN,NaN
4,2011-11-01,BANCO ATLANTIDA S.A.,2.619950,2.482817,0.105909,NaN,NaN
5,2011-12-01,BANCO ATLANTIDA S.A.,2.230437,2.438933,0.160201,2.412881,0.164701
6,2012-01-01,BANCO ATLANTIDA S.A.,2.682136,2.510841,0.199894,2.426543,0.182145
7,2012-02-01,BANCO ATLANTIDA S.A.,3.232366,2.714979,0.409695,2.598898,0.320949
8,2012-03-01,BANCO ATLANTIDA S.A.,2.989239,2.967914,0.225136,2.703424,0.328832
9,2012-04-01,BANCO ATLANTIDA S.A.,3.347692,3.189766,0.149406,2.850303,0.382589


## Ingenieria de caracteristicas financieras: Crecimiento de cartera

Construccion de caracteristicas:

1. Crecimiento mensual de cartera
2. Crecimiento trimestral de cartera

### Crecimiento de cartera de 1 y 3 meses

In [9]:
# ==========================================
# CRECIMIENTO DE CARTERA BRUTA
# ==========================================

REZAGOS_CARTERA = [1, 3]

for k in REZAGOS_CARTERA:

    # Obtener cartera histórica
    cartera_historica = features_temporales[
        ["banco", "fecha", "cartera_bruta"]
    ].copy()

    # Desplazar fechas k meses hacia adelante
    cartera_historica["fecha"] = (
        cartera_historica["fecha"]
        + pd.DateOffset(months=k)
    )

    # Nombre del rezago
    nombre_lag = f"cartera_bruta_lag_{k}"

    cartera_historica = cartera_historica.rename(
        columns={"cartera_bruta": nombre_lag}
    )

    # Incorporar rezago mediante banco y fecha
    features_temporales = features_temporales.merge(
        cartera_historica,
        on=["banco", "fecha"],
        how="left",
        validate="one_to_one"
    )

    # Cartera actual y anterior
    cartera_actual = features_temporales["cartera_bruta"]
    cartera_anterior = features_temporales[nombre_lag]

    # Evitar divisiones no definidas
    validos = (
        cartera_actual.gt(0)
        & cartera_anterior.gt(0)
    )

    # Calcular crecimiento porcentual
    nombre_crecimiento = f"crecimiento_cartera_{k}m_pct"

    features_temporales[nombre_crecimiento] = (
        (cartera_actual / cartera_anterior - 1) * 100
    ).where(validos)


# ==========================================
# VALIDACIÓN
# ==========================================

columnas_crecimiento = [
    "crecimiento_cartera_1m_pct",
    "crecimiento_cartera_3m_pct"
]

print("Dimensiones:", features_temporales.shape)

print("\nVALORES FALTANTES")
display(
    features_temporales[columnas_crecimiento]
    .isna()
    .sum()
)

print("\nESTADÍSTICAS DESCRIPTIVAS")
display(
    features_temporales[columnas_crecimiento]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ).round(3)
)

print("\nMAYORES CRECIMIENTOS TRIMESTRALES")
display(
    features_temporales.nlargest(
        10,
        "crecimiento_cartera_3m_pct"
    )[
        [
            "fecha",
            "banco",
            "cartera_bruta",
            "crecimiento_cartera_3m_pct"
        ]
    ].round(3)
)

print("\nMAYORES CONTRACCIONES TRIMESTRALES")
display(
    features_temporales.nsmallest(
        10,
        "crecimiento_cartera_3m_pct"
    )[
        [
            "fecha",
            "banco",
            "cartera_bruta",
            "crecimiento_cartera_3m_pct"
        ]
    ].round(3)
)

# Verificaciones estructurales
assert len(features_temporales) == 5972

assert not features_temporales.duplicated(
    ["banco", "fecha"]
).any()

assert (
    features_temporales[
        [
            "target_morosidad_t1_pp",
            "target_morosidad_t2_pp",
            "target_morosidad_t3_pp"
        ]
    ].notna().sum().to_dict()
    == {
        "target_morosidad_t1_pp": 5944,
        "target_morosidad_t2_pp": 5922,
        "target_morosidad_t3_pp": 5900
    }
)

print("\nVALIDACIÓN COMPLETADA")

Dimensiones: (5972, 42)

VALORES FALTANTES


crecimiento_cartera_1m_pct    30
crecimiento_cartera_3m_pct    75
dtype: int64


ESTADÍSTICAS DESCRIPTIVAS


,crecimiento_cartera_1m_pct,crecimiento_cartera_3m_pct
count,5942.000,5897.000
mean,1.131,11.532
std,5.382,599.223
min,-40.652,-55.669
1%,-7.784,-16.101
5%,-3.111,-6.755
25%,-0.223,0.008
50%,0.920,2.733
75%,2.048,5.640
95%,5.310,13.683



MAYORES CRECIMIENTOS TRIMESTRALES


...

,fecha,banco,cartera_bruta,crecimiento_cartera_3m_pct
2337,2004-07-01,BP DELBANK,4.610,45996.000
2339,2004-09-01,BP DELBANK,25.017,733.897
2348,2005-06-01,BP DELBANK,325.172,625.678
2349,2005-07-01,BP DELBANK,439.224,468.268
2340,2004-10-01,BP DELBANK,25.428,451.629
2347,2005-05-01,BP DELBANK,140.478,419.090
2350,2005-08-01,BP DELBANK,562.028,300.083
2346,2005-04-01,BP DELBANK,77.292,192.234
2354,2005-12-01,BP DELBANK,1903.638,180.464
2355,2006-01-01,BP DELBANK,2418.136,176.572



MAYORES CONTRACCIONES TRIMESTRALES


/tmp/ipykernel_7066/2308690531.py:106: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(3)


,fecha,banco,cartera_bruta,crecimiento_cartera_3m_pct
1656,2018-03-01,BP CAPITAL,13911.607,-55.669
1657,2018-04-01,BP CAPITAL,13182.817,-51.338
569,2011-06-01,BP AMAZONAS,42765.661,-39.166
570,2011-07-01,BP AMAZONAS,44314.500,-38.899
1658,2018-05-01,BP CAPITAL,12858.760,-38.875
5650,2010-01-01,BP SOLIDARIO,134372.703,-38.773
5651,2010-02-01,BP SOLIDARIO,134670.376,-38.468
1655,2018-02-01,BP CAPITAL,21036.811,-37.255
558,2010-07-01,BP AMAZONAS,45630.184,-36.871
568,2011-05-01,BP AMAZONAS,43042.876,-34.354



VALIDACIÓN COMPLETADA


Debido a la alta presencia de valores extremos, se procede a realizar una transformacion logaritmica de cartera

### Variacion logaritmica de cartera

In [10]:
# ==========================================
# VARIACIONES LOGARÍTMICAS DE CARTERA
# ==========================================

for k in [1, 3]:

    cartera_actual = features_temporales["cartera_bruta"]

    cartera_anterior = (
        features_temporales[f"cartera_bruta_lag_{k}"]
    )

    # Solamente carteras positivas y finitas
    validos = (
        cartera_actual.gt(0)
        & cartera_anterior.gt(0)
        & np.isfinite(cartera_actual)
        & np.isfinite(cartera_anterior)
    )

    # Variación logarítmica
    features_temporales[f"crecimiento_log_cartera_{k}m"] = (
        np.log(
            cartera_actual.where(validos)
            / cartera_anterior.where(validos)
        )
    )


# ==========================================
# COMPROBACIONES
# ==========================================

columnas_log = [
    "crecimiento_log_cartera_1m",
    "crecimiento_log_cartera_3m"
]

print("DIMENSIONES:")
print(features_temporales.shape)

print("\nVALORES FALTANTES:")
display(
    features_temporales[columnas_log].isna().sum()
)

# Examinar exclusivamente el período de desarrollo
revision = features_temporales.loc[
    features_temporales["fecha"] <= "2025-09-01"
].copy()

print("\nESTADÍSTICAS DE DESARROLLO:")
display(
    revision[columnas_log]
    .describe(percentiles=[0.01, 0.05, 0.95, 0.99])
    .round(4)
)

print("\nCRECIMIENTOS TRIMESTRALES EXTREMOS:")
display(
    revision.nlargest(
        10, "crecimiento_cartera_3m_pct"
    )[
        [
            "fecha",
            "banco",
            "cartera_bruta_lag_3",
            "cartera_bruta",
            "crecimiento_cartera_3m_pct",
            "crecimiento_log_cartera_3m"
        ]
    ].round(4)
)

# Validaciones estructurales
assert len(features_temporales) == 5972

assert not features_temporales.duplicated(
    ["banco", "fecha"]
).any()

assert all(
    np.isfinite(features_temporales[col].dropna()).all()
    for col in columnas_log
)

print("\nVALIDACIÓN COMPLETADA")

DIMENSIONES:
(5972, 44)

VALORES FALTANTES:


crecimiento_log_cartera_1m    30
crecimiento_log_cartera_3m    75
dtype: int64


ESTADÍSTICAS DE DESARROLLO:


,crecimiento_log_cartera_1m,crecimiento_log_cartera_3m
count,5666.0000,5621.0000
mean,0.0101,0.0311
std,0.0443,0.1275
min,-0.5218,-0.8135
1%,-0.0810,-0.1776
5%,-0.0313,-0.0708
95%,0.0517,0.1273
99%,0.1133,0.2630
max,0.9970,6.1333



CRECIMIENTOS TRIMESTRALES EXTREMOS:


/tmp/ipykernel_7066/180438034.py:72: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(4)


,fecha,banco,cartera_bruta_lag_3,cartera_bruta,crecimiento_cartera_3m_pct,crecimiento_log_cartera_3m
2337,2004-07-01,BP DELBANK,0.0100,4.6096,45996.0000,6.1333
2339,2004-09-01,BP DELBANK,3.0000,25.0169,733.8970,2.1209
2348,2005-06-01,BP DELBANK,44.8094,325.1720,625.6782,1.9819
2349,2005-07-01,BP DELBANK,77.2917,439.2242,468.2682,1.7374
2340,2004-10-01,BP DELBANK,4.6096,25.4279,451.6294,1.7077
2347,2005-05-01,BP DELBANK,27.0623,140.4780,419.0903,1.6469
2350,2005-08-01,BP DELBANK,140.4780,562.0282,300.0827,1.3865
2346,2005-04-01,BP DELBANK,26.4486,77.2917,192.2337,1.0724
2354,2005-12-01,BP DELBANK,678.7470,1903.6381,180.4636,1.0313
2355,2006-01-01,BP DELBANK,874.3247,2418.1356,176.5718,1.0173



VALIDACIÓN COMPLETADA


## Revision del los otros indicadores financieros

In [11]:
# ==========================================
# VALIDACIÓN DE INDICADORES FINANCIEROS
# ==========================================

indicadores = [
    "solvencia",
    "liquidez",
    "roa",
    "roe",
    "cartera_depositos",
    "cobertura"
]

# Revisar solamente el período de desarrollo
revision = features_temporales.loc[
    features_temporales["fecha"] <= "2025-09-01",
    indicadores
].copy()

# Resumen de calidad
resumen_calidad = pd.DataFrame({
    "faltantes": revision.isna().sum(),
    "ceros": (revision == 0).sum(),
    "negativos": (revision < 0).sum(),
    "infinitos": np.isinf(revision).sum(),
    "minimo": revision.min(),
    "p01": revision.quantile(0.01),
    "mediana": revision.median(),
    "p99": revision.quantile(0.99),
    "maximo": revision.max()
})

display(resumen_calidad.round(4))

,faltantes,ceros,negativos,infinitos,minimo,p01,mediana,p99,maximo
solvencia,8,0,0,0,0.0921,0.1002,0.1438,6.871000e-01,1.328900e+00
liquidez,0,13,0,0,0.0000,0.1214,0.3027,1.142000e+00,7.867400e+01
roa,0,5,311,0,-0.2668,-0.0664,0.0111,5.700000e-02,1.919000e-01
roe,0,5,311,0,-1.0411,-0.2519,0.1059,4.428000e-01,1.207500e+00
cartera_depositos,0,12,0,0,0.0000,0.1576,0.7086,7.772700e+00,4.537340e+01
cobertura,18,0,0,0,0.2427,0.3645,1.4307,3.777322e+06,1.926928e+07


### Identificar cuales son lso valores extremos

In [12]:
# ==========================================
# REVISIÓN FINAL DE VALORES EXTREMOS
# ==========================================

# Exclusivamente período de desarrollo
revision = features_temporales.loc[
    features_temporales["fecha"] < "2025-10-01"
].copy()

# Indicadores que requieren atención
indicadores_extremos = [
    "liquidez",
    "cartera_depositos",
    "cobertura"
]

for indicador in indicadores_extremos:

    print(f"\n{'='*50}")
    print(f"VALORES EXTREMOS: {indicador}")
    print(f"{'='*50}")

    columnas = [
        "fecha",
        "banco",
        indicador,
        "cartera_bruta",
        "cartera_improductiva"
    ]

    display(
        revision.nlargest(10, indicador)[columnas]
    )

# Investigar coberturas extraordinarias
coberturas_extremas = revision.loc[
    revision["cobertura"] > 100
]

print("\nOBSERVACIONES CON COBERTURA > 100:")
print(len(coberturas_extremas))

print("\nINSTITUCIONES INVOLUCRADAS:")
display(
    coberturas_extremas["banco"]
    .value_counts()
)


VALORES EXTREMOS: liquidez


,fecha,banco,liquidez,cartera_bruta,cartera_improductiva
8,2012-03-01,BANCO ATLANTIDA S.A.,78.674004,42649.27620,1274.88896
9,2012-04-01,BANCO ATLANTIDA S.A.,39.842648,43320.19674,1450.22679
11,2012-06-01,BANCO ATLANTIDA S.A.,20.080361,46291.13846,1872.83249
10,2012-05-01,BANCO ATLANTIDA S.A.,9.769577,45151.24750,1636.10543
5853,2016-11-01,BP VISIONFUND ECUADOR S.A.,7.228996,36387.85491,2099.32468
5851,2016-09-01,BP VISIONFUND ECUADOR S.A.,4.710863,36182.06973,2019.71577
12,2012-07-01,BANCO ATLANTIDA S.A.,4.439832,46459.50731,1818.83493
5852,2016-10-01,BP VISIONFUND ECUADOR S.A.,4.321237,36328.87441,1972.17984
5854,2016-12-01,BP VISIONFUND ECUADOR S.A.,4.090894,37148.94890,1999.01385
5002,2003-07-01,BP PROCREDIT,3.536421,14259.14187,110.57038



VALORES EXTREMOS: cartera_depositos


,fecha,banco,cartera_depositos,cartera_bruta,cartera_improductiva
4999,2003-04-01,BP PROCREDIT,45.373416,10250.08348,118.58198
5851,2016-09-01,BP VISIONFUND ECUADOR S.A.,42.582894,36182.06973,2019.71577
5852,2016-10-01,BP VISIONFUND ECUADOR S.A.,36.554885,36328.87441,1972.17984
5002,2003-07-01,BP PROCREDIT,36.532445,14259.14187,110.57038
5000,2003-05-01,BP PROCREDIT,34.595482,11372.38204,122.48004
5853,2016-11-01,BP VISIONFUND ECUADOR S.A.,28.878793,36387.85491,2099.32468
5854,2016-12-01,BP VISIONFUND ECUADOR S.A.,24.547160,37148.94890,1999.01385
5001,2003-06-01,BP PROCREDIT,24.385230,12631.78519,138.25662
5855,2017-01-01,BP VISIONFUND ECUADOR S.A.,20.843809,37735.01099,1987.12180
5856,2017-02-01,BP VISIONFUND ECUADOR S.A.,19.957380,38309.22281,1961.60439



VALORES EXTREMOS: cobertura


,fecha,banco,cobertura,cartera_bruta,cartera_improductiva
2025,2025-03-01,BP CITIBANK,19269278.0,345160.568,0.001
2027,2025-05-01,BP CITIBANK,18914662.0,324940.024,0.001
2026,2025-04-01,BP CITIBANK,18902260.0,348744.544,0.001
2028,2025-06-01,BP CITIBANK,18100213.0,315282.270,0.001
2029,2025-07-01,BP CITIBANK,17566272.0,298606.559,0.001
2030,2025-08-01,BP CITIBANK,16194017.0,308808.021,0.001
2031,2025-09-01,BP CITIBANK,15868067.0,337432.228,0.001
1997,2022-11-01,BP CITIBANK,10814518.0,363390.680,0.002
1998,2022-12-01,BP CITIBANK,10656889.5,371818.798,0.002
1996,2022-10-01,BP CITIBANK,10536460.0,357053.685,0.002



OBSERVACIONES CON COBERTURA > 100:
196

INSTITUCIONES INVOLUCRADAS:


banco
BP CITIBANK        189
BP COOPNACIONAL      7
Name: count, dtype: int64

## Transformaciones logaritmicas para liquidez, y cartera/depositos

In [13]:
# ==========================================
# TRANSFORMACIONES FINANCIERAS FINALES
# ==========================================

variables_transformar = {
    "liquidez": "liquidez_log",
    "cartera_depositos": "cartera_depositos_log"
}

for original, nueva in variables_transformar.items():

    valores = features_temporales[original]

    # Conservar únicamente valores válidos
    validos = (
        valores.ge(0)
        & np.isfinite(valores)
    )

    # Transformación logarítmica
    features_temporales[nueva] = np.log1p(
        valores.where(validos)
    )


# ==========================================
# VALIDACIÓN FINAL
# ==========================================

print("Dimensiones:", features_temporales.shape)

print("\nValores faltantes:")
display(
    features_temporales[
        list(variables_transformar.values())
    ].isna().sum()
)

# Validaciones de integridad
assert len(features_temporales) == 5972

assert not features_temporales.duplicated(
    ["banco", "fecha"]
).any()

targets_esperados = {
    1: 5944,
    2: 5922,
    3: 5900
}

for h, cantidad in targets_esperados.items():

    columna = f"target_morosidad_t{h}_pp"

    assert (
        features_temporales[columna].notna().sum()
        == cantidad
    )

print("\nVALIDACIÓN ESTRUCTURAL CORRECTA")


# ==========================================
# GUARDAR DATASET DEFINITIVO
# ==========================================

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURES_PATH = (
    PROCESSED_DIR
    / "panel_features_multihorizonte.parquet"
)

features_temporales.to_parquet(
    FEATURES_PATH,
    index=False
)

print("\nDataset guardado en:")
print(FEATURES_PATH)

Dimensiones: (5972, 46)

Valores faltantes:


liquidez_log             0
cartera_depositos_log    0
dtype: int64


VALIDACIÓN ESTRUCTURAL CORRECTA

Dataset guardado en:
/home/gustavo-paredes/Documents/Developer/Yachay/morosidad-bancos-ecuador/data/processed/panel_features_multihorizonte.parquet


## Conclusiones de Feature Engineering

La ingeniería de características tuvo como propósito transformar el panel histórico de indicadores financieros de bancos ecuatorianos en un conjunto de datos adecuado para desarrollar modelos supervisados de regresión.
El problema se formuló como una predicción multihorizonte de la morosidad bancaria:
$$
\boxed{
\hat{M}_{b,t+h}=f_h(X_{b,t})
}
$$
Donde:
- b representa la institución bancaria.
- t representa el mes de referencia.
- $(h\in\{1,2,3\})$ representa el horizonte de predicción.
- $(X_{b,t})$ contiene los indicadores disponibles hasta el mes \(t\).
- $(\hat{M}_{b,t+h})$ representa la morosidad estimada para el mes futuro.
Se adoptó una estrategia de predicción directa, mediante la cual se entrenará un modelo independiente para cada horizonte.
Esta estrategia permite evaluar la capacidad predictiva de los modelos a uno, dos y tres meses, sin utilizar predicciones intermedias como entradas de los horizontes posteriores.

4.2. Dataset original

El proceso comenzó con un panel histórico de 23 instituciones bancarias, que contiene información financiera mensual desde enero de 2003 hasta septiembre de 2026.
El dataset original presenta:

| Característica | Resultado |
|---|---:|
| Instituciones | 23 |
| Observaciones | 5.972 |
| Variables originales | 11 |
| Frecuencia | Mensual |
| Inicio | Enero de 2003 |
| Final | Septiembre de 2026 |
| Estructura | Panel desbalanceado |

El panel es desbalanceado porque las instituciones no cuentan necesariamente con la misma cantidad de observaciones históricas.
Se conservaron las observaciones originales y se generaron las características respetando la identidad de cada institución y la continuidad del calendario mensual.
No se eliminaron registros únicamente porque no dispusieran de suficiente información histórica para construir determinados rezagos.

4.3. Construcción de las variables objetivo

Para abordar los tres horizontes predictivos, se construyeron las siguientes variables:

$$\boxed{
\begin{aligned}
y_{t+1}&=M_{t+1}\\
y_{t+2}&=M_{t+2}\\
y_{t+3}&=M_{t+3}
\end{aligned}
}$$

Los indicadores originales de morosidad se encontraban expresados como proporciones, por lo que se multiplicaron por 100 para trabajar directamente en unidades porcentuales.
Por ejemplo:

$$
0.045\longrightarrow4.5\%
$$

Esta decisión facilita la interpretación de las predicciones y permite expresar los errores mediante puntos porcentuales.
Metodología
Los targets se construyeron mediante desplazamientos temporales y uniones por institución y fecha.
De esta manera, cada registro se relaciona exactamente con la observación correspondiente al mes futuro que queremos predecir.
También se verificó que no existieran duplicados por institución y fecha.
Resultados:

| Horizonte | Targets disponibles | Targets faltantes |
|---|---:|---:|
| 1 mes | 5.944 | 28 |
| 2 meses | 5.922 | 50 |
| 3 meses | 5.900 | 72 |

El número de targets disponibles disminuye conforme aumenta el horizonte, debido a que no disponemos de observaciones posteriores a septiembre de 2026.
Los valores faltantes de las variables objetivo se conservaron sin imputación.
Conclusión: se construyeron correctamente tres objetivos de regresión, correspondientes a predicciones de morosidad a uno, dos y tres meses.

4.4. Construcción de rezagos históricos
El EDA mostró que la morosidad presenta una fuerte dependencia temporal.
La autocorrelación mediana de primer orden entre instituciones fue:

$$
\rho_1=0.9133
$$
Incluso con un rezago de doce meses encontramos:
$$

\rho_{12}=0.6549
$$

Por esta razón, construimos características basadas en la morosidad histórica:

$$
\boxed{
M_t,\ M_{t-1},\ M_{t-2},\ M_{t-3},\ M_{t-6},\ M_{t-12}
}
$$

Metodología

Los rezagos se calcularon por institución utilizando fechas calendario.
Esto permitió evitar dos problemas:
Primero, mezclar accidentalmente información histórica de bancos diferentes.
Segundo, tratar observaciones separadas por varios meses como si fueran consecutivas.
Se conservaron como valores faltantes aquellos rezagos para los cuales no existía información histórica suficiente.

Conclusión

Los rezagos permiten representar la persistencia temporal de la morosidad y proporcionar a los modelos información sobre su trayectoria histórica.
La inclusión de varios rezagos busca evaluar si los modelos pueden aprovechar información adicional a la morosidad actual.
No obstante, su contribución efectiva deberá comprobarse durante la validación cruzada temporal.
4.5. Variaciones históricas de morosidad
El EDA permitió identificar que la mayoría de las variaciones mensuales es relativamente pequeña.
Encontramos:

$$
\text{Variación absoluta mediana}=0.2217\text{ pp}
$$
$$

\text{Variación absoluta media}=0.5429\text{ pp}
$$

Sin embargo, también identificamos episodios con cambios superiores a cinco puntos porcentuales.
Por esta razón, decidimos incorporar variables que representen los cambios recientes.

Variación mensual

$$\boxed{
\Delta M_{1,t}=M_t-M_{t-1}
}
$$

Variación trimestral

$$
\boxed{
\Delta M_{3,t}=M_t-M_{t-3}
}$$

Estas características permiten representar incrementos o disminuciones recientes de morosidad.
Se utilizaron diferencias en puntos porcentuales en lugar de crecimientos porcentuales relativos.
Esto evita divisiones por valores de morosidad iguales o cercanos a cero.

Conclusión

Las variaciones históricas complementan los niveles de morosidad, permitiendo distinguir entre instituciones con niveles similares, pero con trayectorias recientes diferentes.
Su incorporación está respaldada por los patrones de autocorrelación encontrados en las variaciones mensuales.

4.6. Estadísticas móviles

Para representar el comportamiento reciente de la morosidad, se construyeron medias móviles y desviaciones estándar móviles.
Media móvil de tres meses

$$
\boxed{
MA_{3,t}=\frac{M_t+M_{t-1}+M_{t-2}}{3}
}
$$
Media móvil de seis meses
$$
\boxed{
MA_{6,t}=\frac{1}{6}\sum_{i=0}^{5}M_{t-i}
}
$$
Adicionalmente, se calcularon:
$$
\sigma_{3,t}
$$
$$
\sigma_{6,t}
$$

Estas variables representan la variabilidad de la morosidad durante los últimos tres y seis meses, respectivamente.

Metodología

Las estadísticas se calcularon de forma independiente para cada institución.
Se utilizó una frecuencia mensual y se exigieron ventanas completas.
Esto significa que una media móvil de seis meses solamente se calculó cuando existían seis observaciones mensuales válidas.
Los resultados fueron:

| Variable | Valores faltantes |
|---|---:|
| Media móvil de 3 meses | 53 |
| Desviación móvil de 3 meses | 53 |
| Media móvil de 6 meses | 122 |
| Desviación móvil de 6 meses | 122 |

Estos valores faltantes son compatibles con los inicios de las series y los períodos históricos con morosidad no verificable.

Conclusión

Las medias móviles proporcionan información sobre el nivel reciente de morosidad, mientras que las desviaciones estándar permiten caracterizar su estabilidad o volatilidad.
Estas características podrían ayudar a los modelos a distinguir períodos de estabilidad de episodios de cambios importantes.

4.7. Características estacionales

Uno de los hallazgos más importantes del EDA fue la identificación de patrones estacionales.
Encontramos que enero presenta incrementos recurrentes de morosidad y diciembre reducciones pronunciadas.
En particular:

$$
\widetilde{\Delta M}_{enero}=+0.3222\text{ pp}
$$
$$
\widetilde{\Delta M}_{diciembre}=-0.4256\text{ pp}
$$
La comparación entre períodos históricos también mostró que estos patrones conservaron su dirección.
Por este motivo, incorporamos características estacionales basadas en el mes del objetivo.

Transformaciones utilizadas
$$
\boxed{
\operatorname{mes}_{\sin,h}
=
\sin\left(\frac{2\pi m_{t+h}}{12}\right)
}
$$
$$
\boxed{
\operatorname{mes}_{\cos,h}
=
\cos\left(\frac{2\pi m_{t+h}}{12}\right)
}
$$

Donde $(m_{t+h})$ representa el mes calendario que queremos predecir.
Por ejemplo, si utilizamos información de septiembre:

Horizonte	Mes objetivo
1 mes	Octubre
2 meses	Noviembre
3 meses	Diciembre


La codificación cíclica permite representar la continuidad entre diciembre y enero.
Conclusión

La inclusión de características estacionales está justificada por los patrones identificados durante el análisis exploratorio.
Como el calendario futuro es conocido de antemano, estas características pueden utilizarse sin introducir información futura sobre la variable objetivo.
Su utilidad predictiva deberá evaluarse mediante validación temporal.

4.8. Crecimiento de cartera bruta

Durante el EDA encontramos que Banco Capital experimentó una contracción importante de su cartera bruta durante el período de elevada morosidad de 2018–2019.
Por esta razón, se decidió incorporar características relacionadas con la evolución del volumen crediticio.
Inicialmente se calcularon:

Crecimiento mensual
$$
g_{1,t}=
\left(\frac{CB_t}{CB_{t-1}}-1\right)100
$$
Crecimiento trimestral
$$
g_{3,t}=
\left(\frac{CB_t}{CB_{t-3}}-1\right)100
$$

Sin embargo, encontramos valores extremadamente elevados, especialmente en los primeros períodos de Banco Delbank.
El máximo crecimiento trimestral alcanzó aproximadamente:
$$
45996\%
$$

Este resultado se originó por una cartera anterior extraordinariamente pequeña.
Para evitar que estas magnitudes dominaran innecesariamente algunos modelos, se construyeron transformaciones logarítmicas.

Transformación definitiva
$$
\boxed{
r_{k,t}=
\ln\left(\frac{CB_t}{CB_{t-k}}\right)
}
$$
Para:
$$
k\in\{1,3\}
$$
Los resultados fueron:

| Estadístico | Crecimiento log. mensual | Crecimiento log. trimestral |
|---|---:|---:|
| Media | 0,0101 | 0,0311 |
| Desviación estándar | 0,0443 | 0,1275 |
| Máximo | 0,9970 | 6,1333 |
| Valores faltantes | 30 | 75 |

Los cálculos se realizaron únicamente cuando ambas carteras eran positivas.
Los casos con denominadores nulos o no válidos se representaron mediante valores faltantes.

Conclusión

Las transformaciones logarítmicas permiten representar los cambios proporcionales de cartera en una escala numérica más manejable.
No eliminan los valores extremos, pero reducen su magnitud relativa y permiten conservar las observaciones históricas.
Para los modelos iniciales se utilizarán las versiones logarítmicas del crecimiento de cartera, evitando incorporar simultáneamente las tasas porcentuales originales.

4.9. Tratamiento de indicadores financieros

También se revisaron los indicadores originales de solvencia, liquidez, rentabilidad, cartera/depósitos y cobertura.
El análisis permitió identificar diferencias importantes en sus distribuciones.
Solvencia, ROA y ROE
Se decidió conservar estos indicadores en su escala original.
Los valores negativos de ROA y ROE son financieramente posibles, ya que pueden representar pérdidas.
Por tanto, no fueron eliminados.
Liquidez y cartera/depósitos
Se identificaron valores extremos elevados en ambas variables.
Para reducir su asimetría numérica, se aplicó:

$$
\boxed{X_{\log}=\ln(1+X)}
$$
Esta transformación admite valores iguales a cero y conserva el orden de las observaciones.
Las nuevas columnas fueron:

liquidez_log
cartera_depositos_log

No se generaron valores faltantes adicionales.

Cobertura

La cobertura presentó valores extraordinariamente elevados.
Se identificaron 196 observaciones superiores a 100, concentradas principalmente en Citibank.
Estas observaciones coincidían con valores de cartera improductiva extremadamente pequeños.
Dado que la cobertura utiliza la cartera improductiva como denominador, estos resultados pueden producirse matemáticamente.
Sin embargo, su distribución presenta valores extremos que podrían afectar algunos algoritmos.
Por esta razón, se decidió excluir cobertura del primer experimento predictivo, conservándola en el dataset para posibles comparaciones posteriores.

Conclusión

Los indicadores financieros fueron tratados respetando su interpretación económica y evitando eliminar observaciones sin una justificación suficiente.
Las transformaciones aplicadas permiten reducir problemas asociados con escalas muy diferentes y valores extremos.
La imputación y el escalado de los predictores se realizarán posteriormente dentro de los pipelines de Machine Learning.

4.10. Incorporación de la institución bancaria

Nuestro EDA evidenció diferencias importantes entre los niveles históricos de morosidad de las instituciones.
Por este motivo, se decidió conservar el identificador banco como una característica categórica.
Durante el entrenamiento se utilizará:
OneHotEncoder(handle_unknown="ignore")
Esto permitirá representar las instituciones sin asignarles un orden numérico artificial.
La codificación se realizará dentro del pipeline para que sea ajustada exclusivamente con las observaciones de entrenamiento de cada fold.

4.11. Separación temporal de desarrollo y Test

Se reservó como período de evaluación final:
$$
\boxed{
\text{Octubre 2025–Septiembre 2026}
}
$$
Los conjuntos se definieron utilizando la fecha objetivo de cada horizonte.
También se consideró la fecha de emisión de las predicciones para evitar evaluar pronósticos que habrían requerido información posterior al momento de su emisión.
Los resultados iniciales fueron:

| Horizonte | Desarrollo | Test |
|---|---:|---:|
| 1 mes | 5.668 | 276 |
| 2 meses | 5.646 | 253 |
| 3 meses | 5.624 | 230 |

Los tres horizontes tendrán períodos de evaluación ligeramente diferentes.
Para facilitar la comparación, posteriormente también se evaluarán sobre un período objetivo común: Diciembre 2025 -> Septiembre 2026
Durante la selección de modelos y la optimización de hiperparámetros se utilizará exclusivamente el período de desarrollo.
El conjunto Test permanecerá reservado hasta finalizar dichas decisiones.

4.12. Resultado final de la ingeniería de características
Después de construir los tres targets y las diferentes características, obtuvimos:

| Característica | Resultado |
|---|---:|
| Observaciones | 5.972 |
| Instituciones | 23 |
| Columnas totales | 46 |
| Horizontes | 1, 2 y 3 meses |
| Estructura | Panel mensual desbalanceado |
| Variable objetivo | Morosidad en puntos porcentuales |
| Dataset final | panel_features_multihorizonte.parquet |

El dataset contiene columnas auxiliares, variables originales, características derivadas y los tres objetivos.
No utilizaremos automáticamente las 46 columnas como predictores.
Seleccionaremos explícitamente las características que correspondan a cada horizonte.

Conclusión general de la ingeniería de características

La ingeniería de características permitió transformar el panel histórico de indicadores financieros bancarios en un conjunto de datos supervisado para la predicción de morosidad a uno, dos y tres meses.
Se construyeron variables objetivo alineadas temporalmente, rezagos históricos de morosidad, variaciones mensuales y trimestrales, estadísticas móviles y características estacionales. Estas transformaciones se fundamentaron en los hallazgos del análisis exploratorio, especialmente en la fuerte persistencia temporal de la morosidad y en los patrones estacionales identificados durante determinados meses del año.
Adicionalmente, se incorporaron características relacionadas con el crecimiento de cartera bruta y se aplicaron transformaciones logarítmicas para representar adecuadamente variables con distribuciones asimétricas y valores extremos. Los indicadores financieros originales también fueron revisados para identificar valores faltantes, ceros, valores negativos y observaciones extraordinarias.
Se conservaron las observaciones históricas financieramente válidas y se evitó imputar las variables objetivo. Las características temporales se calcularon utilizando exclusivamente información correspondiente al mes actual o a períodos anteriores, respetando la identidad de cada institución y la continuidad mensual.
Como resultado, se obtuvo un dataset de 5.972 observaciones y 46 columnas, con tres variables objetivo correspondientes a los horizontes de uno, dos y tres meses.
Para la siguiente fase se utilizarán pipelines de preprocesamiento y validación cruzada temporal, con el propósito de comparar modelos de regresión frente a un baseline de persistencia. El conjunto Test permanecerá reservado para la evaluación final.
La principal hipótesis del proyecto es que las características temporales, estacionales y financieras podrían aportar información adicional para mejorar la predicción de morosidad futura respecto de utilizar únicamente su último valor conocido.

